In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

In [ ]:
# ── Config ──────────────────────────────────────────────
DATA_DIR = "../real_vs_fake_face_dataset/real_vs_fake/real-vs-fake"
BATCH_SIZE = 32
EPOCHS     = 10
LR         = 3e-4
IMG_SIZE   = 224
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

# ── Limit GPU memory to 70% ─────────────────────────────
if DEVICE.type == "cuda":
    torch.cuda.set_per_process_memory_fraction(0.7, device=DEVICE.index)
    
# ── Compression to match  ─────────────────────────────
from io import BytesIO

class RandomJPEGCompression:
    def __init__(self, quality_range=(50, 95)):
        self.quality_range = quality_range
    def __call__(self, img):
        quality = torch.randint(*self.quality_range, (1,)).item()
        buffer = BytesIO()
        img.save(buffer, format="JPEG", quality=quality)
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")

# ── Transforms ──────────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    RandomJPEGCompression(quality_range=(50, 95)),  # ← add this line
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# ── Transforms ──────────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),                      # reduces overfitting
    transforms.ColorJitter(brightness=0.2, contrast=0.2),   # accounts for lighting changes
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],   # ImageNet mean
                         [0.229, 0.224, 0.225]),   # ImageNet std
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# ── Datasets & Loaders ──────────────────────────────────
train_ds = datasets.ImageFolder(f"{DATA_DIR}/train", transform=train_transforms)
val_ds   = datasets.ImageFolder(f"{DATA_DIR}/valid", transform=val_transforms)
test_ds  = datasets.ImageFolder(f"{DATA_DIR}/test",  transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Classes: {train_ds.classes}")   # ['fake', 'real']

# ── Model ───────────────────────────────────────────────
model = models.efficientnet_b0(weights="IMAGENET1K_V1")
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(DEVICE)

# ── Training Setup ──────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:

# ── Train & Eval Functions ──────────────────────────────
def train_epoch(model, loader):
    model.train()
    total_loss, correct = 0, 0
    for imgs, labels in tqdm(loader, desc="Training"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def eval_epoch(model, loader):
    model.eval()
    total_loss, correct = 0, 0
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Evaluating"):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / len(loader), correct / len(loader.dataset), auc

# ── Training Loop ───────────────────────────────────────
best_auc = 0
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss, train_acc = train_epoch(model, train_loader)
    val_loss, val_acc, val_auc = eval_epoch(model, val_loader)
    scheduler.step()

    print(f"Train — Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
    print(f"Val   — Loss: {val_loss:.4f}  | Acc: {val_acc:.4f} | AUC: {val_auc:.4f}")

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), "best_model2.pth")
        print(f"  ✅ Saved best model (AUC: {best_auc:.4f})")

# ── Final Test Evaluation ────────────────────────────────
model.load_state_dict(torch.load("best_model2.pth"))
test_loss, test_acc, test_auc = eval_epoch(model, test_loader)
print(f"\nTest Results — Acc: {test_acc:.4f} | AUC: {test_auc:.4f}")